# Wave 5 + 6i — four gross-code LPU operations, fail-fast

Short report on the Wave-5 campaigns run on BB(12) = [[144,12,12]] (Tour-de-Gross LPU), plus the
Wave-6i inter-module gate. All four operations now have production sweeps:

| run | outdir | status |
|---|---|---|
| **LPU idle** — bare LPU held for `rounds` QEC cycles | `runs/framework/bb144/lpu_idle` | complete 2026-07-24 12:57 |
| **Shift automorphism** — the 14-timestep swap circuit, `C` repeats, `delta = y` | `runs/framework/bb144/automorphism` | complete 2026-07-24 21:08 |
| **Joint Pauli** — `Ybar_1` measured through the whole LPU | `runs/framework/bb144/joint_pauli` | complete 2026-07-25 01:10 |
| **Inter-module** — `Xbar_1 (x) Xbar_1` across two modules via the code-code adapter | `inter_module_r1` / `_r10` (rodan → `runs/cluster/framework/bb144/`) | **complete 2026-07-29** (both legs) |

Common setup: `p_ref = 5e-3`, `adaptive_shots_max = 3000`, full-DEM (`-XYZ`, non-CSS)
**Relay-BP with the tuned-cheap `num_sets=20`** — *not* the paper's 600. The Wave-5 runs use
seed 42 on the local box; the inter-module legs ran on rodan (detached podman, 24 threads each,
frozen weight blocks r1 [1, 1674] / r10 [1, 1742], coupler noise ×1 / ×10). These are fail-fast
numbers: an upper bound on error, a lower bound on what the operation can do.

**Inter-module data comes in two kinds.** The production legs (r1/r10) are on the same footing as
the Wave-5 rows everywhere they appear, with one difference flagged throughout: they carry a
**single observable** (K = 1, `lpu_include_memory_obs` false), so their saturation is 0.5, not
the K = 12 line. The C = 3 / `d_init` = 2 *validation* measurements survive only in §5, as the
historical record of the retired `obs0` "floor" — never to be read against production numbers.

In [ ]:
# src/ is an editable install (`pip install -e .`) - modules import directly, no sys.path needed.
import json
import pathlib
import numpy as np
import matplotlib.pyplot as plt

from lambda_analysis import (load_run, fill_spectrum, reweight_filled, rw_stats, eps_stats,
                             mass_window_p_max, zero_bin_fraction)

RUNS = pathlib.Path("../../runs/framework/bb144")
OPS = {"lpu_idle": "LPU idle", "automorphism": "shift automorphism", "joint_pauli": "Y1 joint Pauli"}
colors = {"lpu_idle": "#2c7fb8", "automorphism": "#d95f0e", "joint_pauli": "#756bb1",
          "inter_module": "#2ca25f"}

runs = {k: load_run(RUNS / k) for k in OPS}
filled = {k: fill_spectrum(r.spectrum) for k, r in runs.items()}
P_TARGET = 1e-3          # the campaign's reporting point
print({k: (len(r.spectrum.weights), r.cycles) for k, r in runs.items()})

# Wave 6i inter-module validation (section 5). Lives in experiments/ rather than runs/,
# which is gitignored, so this notebook renders with no production run present.
W6VAL = json.loads(
    pathlib.Path("../../experiments/tour_de_gross/data/wave6i_intermodule_validation.json")
    .read_text(encoding="utf-8"))
IM_CYCLES = 10           # cycles_of routes inter_module -> lpu_C, and both configs set lpu_C: 10

# Wave 6i PRODUCTION legs (r1/r10), pulled mid-run from rodan into runs/cluster/ (the
# doubled bb144/bb144 is a rsync trailing-slash artifact of the first pull). Loaded ONCE
# here so the section-2 spectrum figure and section 5 read the same objects; None = not
# pulled yet, and every consumer degrades gracefully. NB each leg has its OWN outdir
# (post race-fix); the bare inter_module dir is the stale pre-fix artifact - never read it.
IM_CAND = [RUNS, pathlib.Path("../../runs/cluster/framework/bb144/bb144"),
           pathlib.Path("../../runs/cluster/framework/bb144")]

def _find_im(name):
    for c in IM_CAND:
        if (c / name / "spectrum.json").exists():
            return c / name
    return None

im_runs = {short: (load_run(p) if (p := _find_im(f"inter_module_{short}")) is not None else None)
           for short in ("r1", "r10")}
print({f"inter_module_{k}": (len(r.spectrum.weights) if r else "not pulled")
       for k, r in im_runs.items()})

## 1. Summary

`eps` is per-cycle under `cycles_of`: QEC rounds for idle (12), the repeated-measurement rounds
`lpu_C` for the automorphism, `Ybar_1` and the inter-module legs (10 each). `headroom` is the
rule-of-three exposure from sampled-but-empty bins — reweighted values are lower bounds, headroom
is how far they could rise.

The two **inter-module** rows are now production numbers (complete 2026-07-29), on the same
estimator as the Wave-5 rows. Two flagged differences: they carry a single observable (K = 1 —
fewer ways to fail than the K = 12 rows, so cross-op LER comparisons carry that caveat), and
`D`/`w0` stay `-` because Technique II remains out of reach on these deformed non-CSS circuits.
The r1 sweep descended to w = 31 and r10 to w = 13 before the stop rule fired; the skipped head
below that enters the LER as exactly zero (see §5's health gate for why that is defensible here).

In [ ]:
hdr = f"{'op':<20}{'N_exp':>10}{'cyc':>5}{'D':>5}{'w0':>5}{'LER(1e-3)':>12}{'+-se':>10}{'head':>10}{'eps':>11}{'zero%':>7}"
print(hdr); print("-" * len(hdr))
for k, label in OPS.items():
    r, s = runs[k], filled[k]
    L, se, head = rw_stats(s, P_TARGET)
    eps = eps_stats(s, P_TARGET, r.cycles)[0]
    D = r.distance["distance"] if r.distance else None
    w0 = r.distance["onset"] if r.distance else None
    print(f"{label:<20}{r.spectrum.n_expanded:>10d}{r.cycles:>5d}"
          f"{(D if D else '-'):>5}{(w0 if w0 else '-'):>5}"
          f"{L:>12.2e}{se:>10.1e}{head:>10.1e}{eps:>11.2e}{100*zero_bin_fraction(s):>7.0f}")

# Wave 6i inter-module production legs (complete 2026-07-29). Rows come from the swept
# spectra when pulled (im_runs, setup cell); Technique II stays out of reach on these
# deformed non-CSS circuits, so D/w0 remain '-'. Fallback: sized-not-swept placeholder.
for leg in ("r1", "r10"):
    rr = im_runs.get(leg)
    sz = W6VAL["production_sizing"][leg]
    if rr is None:
        print(f"{'inter-module ' + leg + ' *':<20}{sz['n_expanded']:>10d}{IM_CYCLES:>5d}"
              f"{'-':>5}{'-':>5}{'-':>12}{'-':>10}{'-':>10}{'-':>11}{'-':>7}")
        continue
    sfi = fill_spectrum(rr.spectrum)
    L, se, head = rw_stats(sfi, P_TARGET)
    eps = eps_stats(sfi, P_TARGET, IM_CYCLES)[0]
    print(f"{'inter-module ' + leg:<20}{rr.spectrum.n_expanded:>10d}{IM_CYCLES:>5d}"
          f"{'-':>5}{'-':>5}{L:>12.2e}{se:>10.1e}{head:>10.1e}{eps:>11.2e}"
          f"{100*zero_bin_fraction(sfi):>7.0f}")
if any(im_runs.get(l) is None for l in ("r1", "r10")):
    print("  * sized, not swept (checkpoint not pulled): weights_range frozen to "
          f"[1,{W6VAL['production_sizing']['r1']['w_hi']}] / "
          f"[1,{W6VAL['production_sizing']['r10']['w_hi']}].")
else:
    print("  inter-module sweeps descended to w=31 (r1) / w=13 (r10); the stop-rule-skipped "
          "head (w<=25 / w<=7) enters the LER as exactly zero - see section 5's gate.")

print("\nTechnique-I ansatz (f5, all unpinned):")
for k, label in OPS.items():
    a = runs[k].ansatz["params"]
    print(f"  {label:<20} w0={a['w0']:>5.1f}  f0={a['f0']:.2e}  g1={a['gamma1']:>5.2f}  "
          f"g2={a['gamma2']:>5.2f}  wc={a['wc']:>6.1f}   (cost {runs[k].ansatz['cost']:.1f}, "
          f"{runs[k].ansatz['n_points']} pts)")
for leg in ("r1", "r10"):
    rr = im_runs.get(leg)
    if rr is None or not rr.ansatz:
        print(f"  {'inter-module ' + leg:<20} no stored fit (sweep not complete/pulled)")
        continue
    a = rr.ansatz["params"]
    print(f"  {'inter-module ' + leg:<20} w0={a['w0']:>5.1f}  f0={a['f0']:.2e}  "
          f"g1={a['gamma1']:>5.2f}  g2={a['gamma2']:>5.2f}  wc={a['wc']:>6.1f}   "
          f"(cost {rr.ansatz['cost']:.1f}, {rr.ansatz['n_points']} pts, K=1)")

## 2. The importance-sampled failure spectrum

`f(w)` = fraction of weight-`w` fault configurations the decoder gets wrong — the raw measurement
everything else is built on. The reweighted LER of the next section is just `sum_w f(w) P(w|p)`,
so this is where a sick operation shows itself first.

Points are the sampled bins with binomial error bars; open markers on the floor are
zero-failure bins (plotted at the rule-of-three upper limit `3/T`, i.e. an upper bound, not a
measurement). Lines are the Technique-I `f5` fits. Dashed verticals mark the mean fault weight at
p = 1e-3, `N_exp * q_base * (p/p_ref)` — the weights that actually carry the mass there.

**The small filled green/magenta markers (`r1`/`r10`) are the inter-module production sweep** — the
C = 10 circuit, same footing as the three Wave-5 spectra, drawn from the mid-run checkpoints
pulled from rodan. Partial until the legs finish: the sweep descends from the top of the frozen
block, so the low-weight end fills last. Their dotted f5 lines are **provisional fits, refit
on each pull**: they use the legs single-observable saturation (K = 1, a = 0.5 - not the
Wave-5 K = 12 line) and, until the descent finishes, their fitted w0 tracks the observed
window edge, so the low-w end of the dotted line is an extrapolation, not a measurement. (The
C = 3 validation probe that used to be overlaid here now appears only in §5, as the historical
record of the retired `obs0` "floor" — a different geometry that should never be read against
production spectra.)

In [ ]:
from importance_sampling import failure_spectrum_ansatz, fit_failure_spectrum

fig, ax = plt.subplots(figsize=(6.8, 4.6))
a_sat = 1 - 2.0 ** -12                      # K = 12 saturation, f(w) -> 1 - 2^-K

for k, label in OPS.items():
    r = runs[k]
    w = np.asarray(r.spectrum.weights, float)
    T = np.asarray(r.spectrum.trials, float)
    F = np.asarray(r.spectrum.failures, float)
    f = F / T
    se = np.sqrt(np.clip(f * (1 - f), 0, None) / T)
    hit, zero = F > 0, F == 0
    ax.errorbar(w[hit], f[hit], yerr=se[hit], fmt="o", ms=3.5, lw=.8, color=colors[k],
                label=f"{label}  ({int(hit.sum())}/{len(w)} bins with failures)")
    ax.plot(w[zero], 3.0 / T[zero], "v", ms=3.5, mfc="none", mew=.8, color=colors[k], alpha=.55)

    p = runs[k].ansatz["params"]
    wg = np.linspace(max(p["w0"], 1), w.max(), 400)
    ax.plot(wg, failure_spectrum_ansatz(wg, p["w0"], p["f0"], a_sat, model="f5",
                                        gamma1=p["gamma1"], gamma2=p["gamma2"], wc=p["wc"]),
            "-", lw=1.2, color=colors[k], alpha=.75)
    w_bar = r.spectrum.n_expanded * r.spectrum.q_base * (P_TARGET / r.p_ref)
    ax.axvline(w_bar, color=colors[k], ls="--", lw=.9, alpha=.5)

# --- Wave 6i inter-module PRODUCTION legs (mid-run checkpoints, when pulled) -----------
# The C=10 production sweep, same footing as the three spectra above. Partial until each
# leg finishes: bins shown are final, missing bins have not run yet (the sweep descends,
# so the low-weight end arrives last). The f5 fit is refit HERE on the partial bins
# (nothing is stored until a leg completes) and drawn DOTTED = provisional. Two things
# distinguish these fits from the Wave-5 ones: (a) K = 1 - the legs carry a SINGLE
# observable (lpu_include_memory_obs false), so saturation is a = 1 - 2^-1 = 0.5 (top
# bins measure ~0.49), not the K=12 line; (b) with the low-weight bins still unsampled,
# the fitted w0 tracks the OBSERVED window edge, so the fit's low-w extrapolation is NOT
# trustworthy until the descent finishes (fits track the observable onset). im_fits is
# reused by section 3's ansatz curves. The C=3 validation series lives ONLY in section 5.
IM_STYLE = {"r1": ("#2ca25f", "D"), "r10": ("#c51b8a", "s")}
im_fits = {}
for short, (c_, m_) in IM_STYLE.items():
    rr = im_runs.get(short)
    if rr is None:
        continue
    w = np.asarray(rr.spectrum.weights, float)
    T = np.asarray(rr.spectrum.trials, float)
    F = np.asarray(rr.spectrum.failures, float)
    f = F / T
    hit, zero = F > 0, F == 0
    ax.errorbar(w[hit], f[hit], yerr=np.sqrt(np.clip(f[hit] * (1 - f[hit]), 0, None) / T[hit]),
                fmt=m_, ms=2.8, lw=.7, color=c_,
                label=f"inter-module {short} PRODUCTION (partial, {len(w)} bins)")
    ax.plot(w[zero], 3.0 / T[zero], "v", ms=3, mfc="none", mew=.7, color=c_, alpha=.45)
    w_bar = rr.spectrum.n_expanded * rr.spectrum.q_base * (P_TARGET / rr.p_ref)
    ax.axvline(w_bar, color=c_, ls="--", lw=.9, alpha=.5)
    try:
        fit = fit_failure_spectrum(rr.spectrum, K=1, model="f5", w0=None, f0=None)
        im_fits[short] = fit
        wg = np.geomspace(max(fit.params["w0"], 1), w.max(), 400)
        y = np.asarray(fit.f(wg))
        ax.plot(wg[y > 0], y[y > 0], ":", lw=1.3, color=c_, alpha=.8,
                label=f"inter-module {short} $f5$ (provisional, K=1)")
    except Exception as e:
        print(f"inter-module {short}: f5 fit failed ({e}) - points only")

ax.axhline(a_sat, color="k", ls=":", lw=.8)
ax.text(900, a_sat * 1.1, r"$1-2^{-K}$", fontsize=7, va="bottom", ha="right")
ax.axhline(0.5, color="k", ls=":", lw=.6, alpha=.6)
ax.text(1.3, 0.53, r"$1-2^{-1}$ (inter-module, K=1)", fontsize=6, va="bottom")
ax.set(xscale="log", yscale="log", xlabel="fault weight $w$", ylabel="$f(w)$",
       ylim=(1e-4, 4), xlim=(1, 1800),
       title="Measured failure spectra (points) and $f5$ fits (lines)")
ax.legend(fontsize=6.5, loc="upper left"); ax.grid(alpha=.25, which="both")
fig.tight_layout()

## 3. Logical error rate

Points = importance-sampled spectrum, gap-filled, reweighted; lines = the Technique-I `f5` fit.
Curves are cut at `mass_window_p_max` (4 sigma of binomial mass inside the sampled window) —
beyond it reweighting is no longer unbiased.

**The green/magenta marker-lines are the inter-module production legs (r1/r10)** — the same
reweighted-IS estimator as the Wave-5 points, from the mid-run checkpoints. Their curves are cut
on *both* ends: high p by the usual mass window, and low p because the descending sweep has not
yet sampled the low-weight bins — unlike the Wave-5 runs' skipped bins those are not known-~0, so
the curve is only drawn where the binomial mass sits entirely inside the sampled window. The span
widens with every pull as the sweep descends. The dotted lines are section 2s provisional
K = 1 fits pushed through the ansatz reweighting: trustworthy where they track the markers,
an extrapolation sketch beyond - especially at low p, where the fitted w0 rides the
still-descending window edge.

**The hollow green diamonds and purple crosses are direct Monte Carlo, not importance sampling**,
at C = 3 / `d_init` = 2 — a different estimator on a different circuit depth. They are on this
axis for one reason: they are the measurement that produced, and then retired, the reported
`obs0` "floor". The inter-module gate reads ~0.4 at p = 5e-3 and **0.020 ± 0.010 at p = 1e-3** —
a threshold crossing, not a floor. The `Ybar_1` crosses are the control that settles it: the
*known-good* operation, run at the same settings, floors just as hard at p = 5e-3 (0.35). At
~106 expected faults per shot against `d ~ 10`, LER ~ 0.5 there is arithmetic, not a defect.
Do not compare either against the IS curves.

In [ ]:
from importance_sampling import logical_error_rate_from_ansatz

fig, ax = plt.subplots(figsize=(6.4, 4.4))

for k, label in OPS.items():
    r, s = runs[k], filled[k]
    res = np.load(RUNS / k / "result.npz")
    pg = res["p_values"]
    ok = pg <= mass_window_p_max(s)
    ax.plot(pg[ok], reweight_filled(s, pg[ok]), "o", ms=4, color=colors[k], label=f"{label} (IS)")
    ax.plot(res["ansatz_p"], res["ansatz_P"], "-", lw=1.4, color=colors[k], alpha=.75,
            label=f"{label} (ansatz)")

# --- Wave 6i inter-module PRODUCTION legs: reweighted IS + provisional ansatz ----------
# No result.npz until a leg completes, so the p-grid is built here and the reweighted
# curve is cut on BOTH ends: high p by mass_window_p_max (as above), and low p because
# the mid-run sweep descends - the LOW-weight bins are still unsampled, and unlike the
# Wave-5 runs' skipped below-onset bins they are not known-~0. Quote only where the
# 4-sigma binomial mass sits inside the sampled window [w_min, w_max]. The DOTTED line
# is section 2's provisional K=1 f5 fit pushed through the ansatz reweighting (Eq. 10):
# inside the marker span it should track the points; OUTSIDE (esp. low p) it is an
# extrapolation from a fit whose w0 tracks the observed window edge - read it as a
# lower-bound-flavored sketch until the descent completes, not a prediction.
IM_STYLE3 = {"r1": ("#2ca25f", "D"), "r10": ("#c51b8a", "s")}
for short, (c_, m_) in IM_STYLE3.items():
    rr = im_runs.get(short)
    if rr is None:
        continue
    sf = fill_spectrum(rr.spectrum)
    w_min = min(rr.spectrum.weights)
    pg = np.geomspace(2e-4, 1e-2, 40)
    mu = rr.spectrum.n_expanded * rr.spectrum.q_base * (pg / rr.p_ref)
    ok = (mu - 4 * np.sqrt(mu) >= w_min) & (pg <= mass_window_p_max(sf))
    if ok.any():
        ax.plot(pg[ok], reweight_filled(sf, pg[ok]), m_, ms=4, lw=0, color=c_,
                label=f"inter-module {short} (IS, partial sweep)")
    else:
        print(f"inter-module {short}: no p where the sampled window carries the mass yet")
    if short in im_fits:
        ax.plot(pg, logical_error_rate_from_ansatz(im_fits[short], pg), ":", lw=1.4,
                color=c_, alpha=.8, label=f"inter-module {short} (ansatz, provisional)")

# --- Wave 6i inter-module: DIRECT MONTE CARLO at C=3/d_init=2 --------------------------
# NOT importance-sampled and NOT the production geometry, so these are not comparable to
# the curves above - they are here to show the shape that retired the reported "floor":
# ~0.4 at p=5e-3 (far above threshold, ~106 expected faults/shot) but 0.02 at p=1e-3.
# A single high-p MC point cannot distinguish a broken observable from an over-noised
# circuit; that is what section 5's f(w) is for.
_mc = [q for q in W6VAL["monte_carlo_ler"]["points"] if q["circuit"] == "inter_module"]
_p = np.array([q["p"] for q in _mc], float)
_L = np.array([q["ler"] for q in _mc], float)
_se = np.array([q["se"] for q in _mc], float)
ax.errorbar(_p, _L, yerr=_se, fmt="D", ms=5.5, lw=0, elinewidth=1.1, capsize=3,
            mfc="none", mew=1.4, color=colors["inter_module"],
            label="inter-module (direct MC, C=3 - not IS, not production)")

_y1 = [q for q in W6VAL["monte_carlo_ler"]["points"]
       if q["circuit"] == "y1_baseline" and not q["idle"]]
ax.errorbar([q["p"] for q in _y1], [q["ler"] for q in _y1],
            yerr=[q["se"] for q in _y1], fmt="x", ms=7, lw=0, elinewidth=1.1, capsize=3,
            color=colors["joint_pauli"], alpha=.9,
            label="Ybar_1 same-setup MC control (floors just as hard)")

ax.axvline(P_TARGET, color="k", ls=":", lw=.8)
ax.set(xscale="log", yscale="log", xlabel="physical error rate p", ylabel="logical error rate",
       ylim=(1e-12, 2), title="Wave-5 gross-code LPU operations (relay num_sets=20)")
ax.legend(fontsize=6.5, ncol=1, loc="lower right"); ax.grid(alpha=.25, which="both")
fig.tight_layout()

## 4. What the runs say

**Cost tracks DEM size.** Expanded mechanism counts are 5.2e5 (idle) / 1.7e6 (automorphism) /
2.9e6 (`Ybar_1`), and the LER at p = 1e-3 orders the same way: ~2e-5, ~6e-3, ~0.5. `Ybar_1` is the
whole LPU in one operation, so at fixed p it carries roughly 6x the idle fault load; its shallow
`gamma1 = 4.55` is that, not a decoder pathology. The inter-module gate extends the trend and sits
at the top of it — **4.5e6 / 4.7e6** expanded mechanisms, 1.55x / 1.62x `Ybar_1` — which is what two
modules plus the adapter costs. Its LER is not yet measurable at that geometry, but the ordering
is already fixed by the DEM.

**Idle is the only run with Technique II**: D = 10 (bound), `w0` = 5, `f0*` = 4.8e-15, which meets
the paper's `d <= 10` bound for the LPU idle. Technique II was **dropped** from the automorphism
and `Ybar_1` configs — BP-OSD runs ~2 h/decode at 2e5+ columns (the smoke stalled at 6/50 trials
in 10 h). Their `result.npz` therefore carries `distance = 0`, `onset = nan` **by design**, and
their fits are `w0`/`f0`-free. The paper likewise has no `Ybar_1` row in Table 2. The inter-module
config drops it for the same reason, with an extra one: `compute_distance` returns a spurious 1 on
these deformed non-CSS circuits.

**Both IS sweeps ended on the stop rule, not exhaustion** (3 consecutive zero-failure bins at
`shots_max = 3000`): the automorphism skipped 14 lower weights, `Ybar_1` skipped 1. Skipped
weights sit below the onset and contribute ~0.

### Three caveats before quoting any of this

1. **Never read `is_P_logical` straight out of `result.npz`.** The raw IS sum runs only over
   *sampled* weights, so a strided tail saturates at exactly `1/stride` instead of going to 1:
   idle is stride 1 (-> 1.0), the automorphism stride 4 (-> 0.250), `Ybar_1` stride 6 (-> 0.1667).
   It is low by ~stride at *every* p. `f(w)` itself is healthy (1.0 in the top bins). The cell
   below shows gap-filling closing the gap against the ansatz.
2. **Decoder handicap.** `num_sets = 20`, not the paper's 600. Combined with `p_ref = 5e-3` vs the
   paper's 1e-4 importance-sampling prior, these numbers are directionally comparable to Table 3,
   not numerically.
3. **The inter-module entries are not production numbers.** Only `N_exp` and the frozen weight
   blocks are properties of the C = 10 circuit; everything else shown for it is C = 3 validation.
   Nothing in this section's cost or ordering discussion should be quoted for it beyond the DEM
   size.

In [ ]:
# The stride artifact, and gap-filling as the fix.
print(f"{'op':<20}{'stride':>8}{'raw sat.':>10}{'1/stride':>10}   | at p=1e-3: {'raw':>10}{'filled':>10}{'ansatz':>10}")
for k, label in OPS.items():
    r, s = runs[k], filled[k]
    res = np.load(RUNS / k / "result.npz")
    w = np.asarray(r.spectrum.weights)
    stride = int(np.median(np.diff(w)))
    i = int(np.argmin(np.abs(res["p_values"] - P_TARGET)))
    print(f"{label:<20}{stride:>8d}{res['is_P_logical'][-1]:>10.4f}{1/stride:>10.4f}   |            "
          f"{res['is_P_logical'][i]:>10.2e}{reweight_filled(s, [P_TARGET])[0]:>10.2e}"
          f"{res['ansatz_P'][i]:>10.2e}")

## 5. Inter-module gate — VALIDATED, production complete 2026-07-29

The fourth operation, and the one Wave 6i is built around: a **gross-to-gross adapter** joining two
[[144,12,12]] modules (A frame 0, B frame 378) to measure `Xbar_1 (x) Xbar_1` across them. Merged to
`main`; branch was `wave6-intermodule`.

**Built and green:** `AdapterGraph` + cross-module `U_B` derivation, `build_adapter_cycle` (11 bridge
Bell checks + 10 cross-module `U_B` checks), `build_joint_x1x1_circuit`; wired into
`experiment_runner` and `lambda_analysis`; configs `gross_intermodule_{r1,r10}.yaml`. Gates **E1**
(p=0 determinism), **E2** (obs0 reproduces the MPP reference), **E4** (DEM + decoder) all pass.
Both paper-ambiguous knobs were resolved empirically: the bridge identification is **CX** (CZ
anticommutes with the X-vertex checks), and the cross-module `U_B` check takes **both** physical
copies of each bridge qubit.

### The reported `obs0` floor was a MISDIAGNOSIS — retired 2026-07-27

The earlier draft of this section recorded an open blocker: *"`obs0` floors at LER ~ 0.4, a bridge
Bell gauge issue."* That was a real measurement read the wrong way.

> **A Monte-Carlo LER at `p_ref` cannot distinguish a broken observable from a circuit operating far
> above threshold.** It is not a diagnostic for observable health.

Three independent lines of evidence retire it:

1. **The circuit is far above threshold at that point.** Straight from the DEM it sees **105.7
   expected faults per shot** at p = 5e-3 against `d ~ 10`. LER ~ 0.5 there is arithmetically
   unavoidable for *any* circuit of this size.
2. **The known-good baseline floors just as hard.** `Ybar_1` — validated, in this very report —
   gives LER 0.35 (idle off) and 0.51 (idle on) at the same point.
3. **It decodes fine below threshold.** The inter-module circuit reaches **LER 0.020 ± 0.010 at
   p = 1e-3**. That is a threshold crossing, not a floor.

The correct, **p-independent** health check is the low-weight failure spectrum `f(w)` — the same
quantity section 2 plots, and exactly what `techniques: [IS, I]` measures. It is now a standing gate
in the runbook, alongside the weight-1 degeneracy scan promoted after the `Ybar_1` framing bug: **the
degeneracy scan catches *undetectable* faults, `f(w)` catches *miscorrected* ones.**

Two cautions carried forward, both learned the hard way here:

* **Compare against the validated baseline, never against zero.** `Ybar_1` is itself nonzero at
  w = 3, 4, 6 (1, 1, 3 per 400) — decoder miscorrection at the cheap 20-set relay, not undetectable
  errors. "Matches the baseline" is the standard.
* **`shortest_graphlike_error` is a misleading proxy** on these deformed non-CSS circuits: it skips
  precisely the hyperedge/gauge errors at issue, and `compute_distance` returns a spurious 1 (the
  validated `Ybar_1` circuit does too). Where MC, graphlike distance and `f(w)` disagree, **trust
  `f(w)`.**

### `close_cycles` — correct, but not the cure

A fix landed alongside: terminal Z-cycle closure detectors for both modules' `U_l` cycles and the 10
cross-module `U_B` checks, each XORing the last merged round's cycle record against its edge
readouts — the `prod m_e = +1` boundary `build_joint_pauli_circuit` already used. Without them the
final merged round's Z-cycle information was simply discarded. It adds 20 detectors (2722 -> 2742
here; 10883 -> 10903 at production geometry), all verified deterministic at p = 0.

**Honest scope: it is a correctness improvement, not what removed the floor** — the pre-closure
circuit is equally clean at low weight, as the cells below show. The X-type bridge Bell checks have
no analogue (a Z-basis edge readout cannot cancel them) and still terminate into `obs0`.

### Production outcome (2026-07-29)

Both legs completed on rodan at the frozen blocks (r1 [1, 1674], r10 [1, 1742], stride 6). The
sweeps descended to **w = 31 (r1)** and **w = 13 (r10)** before the 3-consecutive-zero-bins stop
rule fired — far deeper than the mid-run reads suggested — so the reweighted LER is unbiased down
to **p ≈ 2.1e-4 (r1) / 1.2e-4 (r10)**, and the low-weight health gate below is nearly complete.
The gate **passes**: the legs' low-weight bins sit at or below the `Ybar_1` baseline (e.g. w = 37:
0/3000 vs the baseline's 4/3000) — no sub-onset decoder-floor signature, with the K = 1-vs-K = 12
caveat noted in §1. The coupler-noise cost is the headline physics: LER(r10)/LER(r1) grows from
~1.0 at p = 2e-3 to 1.3 at 1e-3 and 1.8 at 5e-4, and the stored fits (r1 w0 = 49, r10 w0 = 31 —
both onsets now well inside the sampled range) project the gap widening steeply below that.
`lpu_include_memory_obs` remains `false` pending the K=23 merged-graph recipe, so these runs carry
the operator observable only.

In [ ]:
import json

# Validation measurements live in experiments/ (not runs/, which is gitignored) so this
# section renders without a production run. Reproduce with:
#   python experiments/tour_de_gross/failure_spectrum_probe.py --op inter_module ...
W6 = pathlib.Path("../../experiments/tour_de_gross/data/wave6i_intermodule_validation.json")
val = json.loads(W6.read_text(encoding="utf-8"))
fs = val["failure_spectrum"]
W = np.asarray(fs["weights"], int)
T = fs["conditions"]["T_per_weight"]

print(f"f(w) validation - C={fs['conditions']['C']} d_init={fs['conditions']['d_init']} "
      f"p={fs['conditions']['p']:g}, {fs['conditions']['decoder']}, T={T}/weight\n")
print(f"{'series':<36}" + "".join(f"w={w:<6d}" for w in W) + "  total")
for key, s in fs["series"].items():
    F = np.asarray(s["failures"], int)
    print(f"{s['label']:<36}" + "".join(f"{f:<8d}" for f in F) + f"  {F.sum()}/{len(W) * T}")

ef = val["expected_faults_per_shot"]
print("\nE[faults/shot] (DEM, decoder-independent) - why MC at p_ref cannot diagnose obs0:")
for p, a, b in zip(ef["p"], ef["inter_module"], ef["y1_baseline"]):
    print(f"  p={p:<8g} inter-module {a:>7.2f}   Y1 baseline {b:>7.2f}")

# Production legs are loaded once in the setup cell (im_runs; per-leg outdirs, local runs/
# first then the cluster pull). The stale pre-race-fix inter_module dir is never read.
_legs = [k for k, r in im_runs.items() if r is not None]
if _legs:
    print(f"\nPRODUCTION checkpoints present: {', '.join(_legs)} - drawn in the section-2 "
          "figure and detailed in the 'Production checkpoints' cells below. Mid-run: the "
          "validation above stands until a leg completes.")
else:
    print("\nNo production checkpoints pulled yet - validation only. Legs launch via "
          "container/run_intermodule.sh on rodan; pull home with "
          "rsync ...:stim_work/runs/framework/bb144/ runs/cluster/framework/bb144/")

In [ ]:
from scipy.stats import beta


def clopper_pearson(k, n, alpha=0.05):
    """Exact binomial CI. Counts here are 0-3 out of 400, where the normal approximation
    used in section 2 is invalid - it collapses to zero width at k=0, hiding all of the
    uncertainty in exactly the bins the conclusion rests on."""
    k = np.asarray(k, float)
    lo = np.where(k > 0, beta.ppf(alpha / 2, np.maximum(k, 1), n - k + 1), 0.0)
    hi = np.where(k < n, beta.ppf(1 - alpha / 2, k + 1, np.maximum(n - k, 1)), 1.0)
    return lo, hi


STYLE = {"y1_baseline": ("#756bb1", "o"),     # same purple as joint_pauli in section 2
         "inter_prefix": ("#999999", "s"),
         "inter_closure": ("#2ca25f", "D")}
SCALE = 1e-3

fig, ax = plt.subplots(figsize=(7.0, 4.2))
for i, (key, s) in enumerate(fs["series"].items()):
    F = np.asarray(s["failures"], float)
    f = F / T
    lo, hi = clopper_pearson(F, T)
    c, m = STYLE[key]
    off = (i - 1) * 0.16                      # nudge: the series share identical values
    ax.errorbar(W + off, f / SCALE, yerr=[(f - lo) / SCALE, (hi - f) / SCALE],
                fmt=m, ms=6, lw=0, elinewidth=1.3, capsize=3.5, color=c,
                label=f"{s['label']}  ({int(F.sum())}/{len(W) * T})")

ax.set(xlabel="fault weight $w$", ylabel=r"$f(w)$  [$\times 10^{-3}$]", xticks=W,
       ylim=(0, 22), xlim=(0.5, 6.5),
       title="Wave 6i: inter-module $f(w)$ vs the validated $\\bar{Y}_1$ baseline\n"
             "points = measured, bars = exact 95% binomial CI (T=400/weight)")
ax.legend(fontsize=7.5, loc="upper left", framealpha=.95)
ax.grid(alpha=.25, axis="y")
fig.tight_layout()

print("Every interval overlaps every other at every weight => indistinguishable.")
print(f"A k=0 bin is NOT 'zero': its 95% CI is [0, {3.0 / T:.4f}] - the rule-of-three bound,")
print(f"which is why a clean 0/{T} BOUNDS the floor at ~{1/T:.1e} rather than disproving one.")
print(f"Ruling out the ~5e-4-class floor the campaign configs reference needs T >~ 10000.")

### Production checkpoints — r1 / r10, read mid-run

Both legs launched on rodan 2026-07-27 (detached podman, 24 threads each, frozen weight blocks
r1 [1, 1674] / r10 [1, 1742]). The cell below reads whatever checkpoint has been **pulled home**
(`rsync` → `runs/cluster/framework/bb144/`) and is safe to re-run after every pull: per-weight
checkpoints are atomic, so a partial `spectrum.json` is a valid subset of the sweep — every bin
shown is final, the rest simply have not run yet.

Reading rules for a mid-run checkpoint, same discipline as section 5:

* **The sweep works downward from the top of the block**, so the mass region (w ≈ 1518–1583 at
  `p_ref`) fills early and the **low-weight sub-onset bins arrive last**. Consequence: the
  reweighted LER becomes quotable well before the run ends, but the *health gate* — the
  low-weight `f(w)` comparison against the baseline — is only complete when the sweep has walked
  all the way down (or stopped on the 3-consecutive-zero-bins rule, skipping the sub-onset
  remainder as the Wave-5 runs did).
* The comparison standard is the `Ybar_1` **production** spectrum (faint purple), the validated
  baseline at comparable geometry — never zero.
* Zero-failure bins are rule-of-three **bounds** (open markers at `3/T`), not measurements.
* The LER print carries its own guard: it appears only when the sampled window carries the
  binomial mass at `p_target` (`mass_window_p_max`); otherwise the cell says why not.

In [ ]:
# Production checkpoints, mid-run — detail view of the legs loaded in the setup cell
# (im_runs). Re-pull (rsync) + re-run the notebook any time for a fresh read.
IM_LEGS = {"r1": ("#2ca25f", "D", 1674), "r10": ("#c51b8a", "s", 1742)}

fig, ax = plt.subplots(figsize=(7.2, 4.6))
# the validated production-geometry baseline: Ybar_1's own sweep, faint, for shape comparison
_y1 = runs["joint_pauli"].spectrum
_yW, _yF, _yT = (np.asarray(a, float) for a in (_y1.weights, _y1.failures, _y1.trials))
_yhit = _yF > 0
ax.plot(_yW[_yhit], (_yF / _yT)[_yhit], ".", ms=3, color=colors["joint_pauli"], alpha=.35,
        label="Ybar_1 production (validated baseline)")

for short, (col, mark, w_hi) in IM_LEGS.items():
    r = im_runs.get(short)
    if r is None:
        print(f"{short}: no checkpoint pulled yet")
        continue
    s = r.spectrum
    W_, F_, T_ = (np.asarray(s.weights, int), np.asarray(s.failures, int),
                  np.asarray(s.trials, int))
    print(f"{short}: {len(W_)} bins of block [1,{w_hi}] done (w={W_.min()}..{W_.max()}), "
          f"{int(T_.sum()):,} shots, {int(F_.sum()):,} failures   "
          f"[{_find_im(f'inter_module_{short}')}]")
    print("   low-w gate: " + "  ".join(f"f({w})={f}/{t}"
                                        for w, f, t in zip(W_[:8], F_[:8], T_[:8])))
    f = F_ / np.maximum(T_, 1)
    hit, zero = F_ > 0, F_ == 0
    ax.errorbar(W_[hit], f[hit], yerr=np.sqrt(np.clip(f[hit] * (1 - f[hit]), 0, None) / T_[hit]),
                fmt=mark, ms=4, lw=.8, color=col,
                label=f"inter-module {short} (partial: {len(W_)} bins)")
    ax.plot(W_[zero], 3.0 / np.maximum(T_[zero], 1), "v", ms=3.5, mfc="none", mew=.8,
            color=col, alpha=.5)
    mu = W6VAL["production_sizing"][short]["mu_p_hi"]
    ax.axvline(mu, color=col, ls="-.", lw=.8, alpha=.5)
    # LER only when the sampled window carries the mass at P_TARGET (mid-run honesty guard)
    sf = fill_spectrum(s)
    p_max = mass_window_p_max(sf)
    if p_max >= P_TARGET:
        L, se, head = rw_stats(sf, P_TARGET)
        print(f"   LER({P_TARGET:g}) = {L:.2e} +-{se:.1e}  (head {head:.1e}; window ok to "
              f"p={p_max:.1e})")
    else:
        print(f"   LER({P_TARGET:g}): not quotable yet - sampled window unbiased only to "
              f"p <= {p_max:.1e} (prefix ends at w={W_.max()} of mu~{mu:.0f})")

ax.axhline(1 - 2.0 ** -12, color="k", ls=":", lw=.8)
ax.set(xscale="log", yscale="log", xlabel="fault weight $w$", ylabel="$f(w)$",
       xlim=(1, 1800), ylim=(1e-4, 4),
       title="Wave 6i production legs, mid-run vs the $\\bar{Y}_1$ baseline\n"
             "(open markers = zero-failure bins at the 3/T bound; dash-dot = each leg's mass at $p_{ref}$)")
ax.legend(fontsize=6.5, loc="upper left")
ax.grid(alpha=.25, which="both")
fig.tight_layout()

## 6. Next

* **⚠️ rodan has NO SCHEDULER.** Discovered 2026-07-27: no `sbatch`/`srun`, and it is a *shared*
  96-core box. `experiments/slurm/submit_lpu.sh` and `submit_lpu_boost.sh` are unusable there —
  launch via detached podman with `container/run_lpu_boost.sh` instead (env-var thread capping, not
  `podman --cpus`; mounts `experiments/` as well as `runs/`, which `container/run_local.sh` does
  not, so that one would silently run the image's baked configs). Budget: 3 boost jobs × 8 threads
  = 24 of 96 cores; adding the two Wave-6i jobs makes 40 of 96.
* **Wave 5b boost pass** re-samples all three at deeper caps to push the zero-bin floor from the
  rule-of-three ~1e-3 toward ~1e-5. That is what buys `Ybar_1` a meaningful low-p tail; the paper's
  `num_sets = 600` relay and a real Technique II remain out of reach at this column count.
* **Wave 6i production run**: needs `container/run_intermodule.sh` written first (a two-line
  addition to the boost launcher), then `r1` + `r10` at the frozen weight blocks.
* **Large-T `f(w)` confirmation** for the inter-module gate — the section-5 result bounds the floor
  at ~2.5e-3, which sits *above* the ~5e-4-class floor the campaign configs reference. `T = 10000`
  closes that gap; `failure_spectrum_probe.py` is chunked, checkpointed per weight, and tops up on
  re-run rather than restarting.
* **Decoder variant**: a `decoder_p` knob (`CalibratedRelayBP`, ported from the K=4 work) as a
  separate config, to separate decoder miscalibration from circuit cost in the `Ybar_1` number.
* **Contiguous low-weight tails** for the automorphism and `Ybar_1` if their low-p ends are ever
  quoted as point values rather than through the fit.